# Feature Engineering

In [145]:
import pandas as pd

In [146]:
cleaned_data = pd.read_parquet("../data/cleaned_data.parquet")
weather_data = pd.read_parquet("../data/weather_data.parquet")

In [147]:
cleaned_data["FL_DATE"] = pd.to_datetime(cleaned_data["FL_DATE"], format="%m/%d/%Y %I:%M:%S %p")

In [148]:
cleaned_data["YEAR"] = cleaned_data["FL_DATE"].dt.year
cleaned_data["MONTH"] = cleaned_data["FL_DATE"].dt.month
cleaned_data["DAY_OF_MONTH"] = cleaned_data["FL_DATE"].dt.day
cleaned_data["DAY_OF_WEEK"] = cleaned_data["FL_DATE"].dt.dayofweek

In [149]:
weather_data['DATE_TIME'] = pd.to_datetime(weather_data['datetime'])
weather_data = weather_data.drop(columns=['datetime'])
weather_data['DATE'] = weather_data['DATE_TIME'].dt.date
weather_data['HOUR'] = weather_data['DATE_TIME'].dt.hour

In [150]:
cleaned_data['DATE'] = cleaned_data['FL_DATE'].dt.date

In [151]:
cleaned_data.shape

(9064182, 32)

In [152]:
for column in weather_data.columns:
  
    print(column, weather_data[column].isna().sum())

iata_code 0
precipitation 0
rain 0
snowfall 0
wind_speed 0
wind_gusts 0
temperature 0
weather_code 0
cloud_cover_low 0
DATE_TIME 0
DATE 0
HOUR 0


In [153]:
cleaned_data[["DAY_OF_WEEK", "MONTH", "DAY_OF_MONTH", "YEAR"]].head()

,DAY_OF_WEEK,MONTH,DAY_OF_MONTH,YEAR
0,0,9,1,2025
1,0,9,1,2025
2,0,9,1,2025
3,0,9,1,2025
4,0,9,1,2025


In [154]:
cleaned_data["DEPARTURE_HOUR"] = cleaned_data["CRS_DEP_TIME"] // 100
cleaned_data["DEPARTURE_MINUTE"] = cleaned_data["CRS_DEP_TIME"] % 100
cleaned_data = cleaned_data.drop(columns=["CRS_DEP_TIME"])

In [155]:
cleaned_data = cleaned_data.merge(
  weather_data[['iata_code', 'DATE', 'HOUR', 'precipitation', 'rain', 
                'snowfall', 'wind_speed', 'wind_gusts', 'temperature', 
                'weather_code', 'cloud_cover_low']],
  left_on=['ORIGIN', 'DATE', 'DEPARTURE_HOUR'],
  right_on=['iata_code', 'DATE', 'HOUR'],
  how='left'
)

In [156]:
cleaned_data = cleaned_data.drop(columns=['iata_code', 'DATE', 'HOUR'])

In [157]:
cleaned_data["ROUTE"] = cleaned_data["ORIGIN"] + "-" + cleaned_data["DEST"]

In [158]:
def isHolidayPeriod(date):
    # MLK Day
    if date.month == 1 and 15 <= date.day <= 20:
        return 1

    # Presidents Day
    if date.month == 2 and 13 <= date.day <= 17:
        return 1
    
    # Spring Break
    if (date.month == 3 and date.day >= 15) or (date.month == 4 and date.day <= 15):
        return 1
    
    # Memorial Day 
    if date.month == 5 and 23 <= date.day <= 27:
        return 1
    
    # Fourth of July
    if date.month == 7 and 1 <= date.day <= 7:
        return 1
    
    # Labor Day
    if (date.month == 8 and date.day >= 29) or (date.month == 9 and date.day <= 2):
        return 1
    
    # Thanksgiving (Nov 20 – Dec 1)
    if (date.month == 11 and date.day >= 20) or (date.month == 12 and date.day <= 1):
        return 1

    # Christmas & New Year (Dec 20 – Jan 5)
    if (date.month == 12 and date.day >= 20) or (date.month == 1 and date.day <= 5):
        return 1

    return 0

In [159]:
cleaned_data['IS_HOLIDAY_PERIOD'] = cleaned_data['FL_DATE'].apply(isHolidayPeriod)

In [160]:
cleaned_data['DELAYED'] = (cleaned_data['DEP_DELAY'] > 15).astype(int)

In [161]:
cleaned_data['ORIGIN_HOURLY_FLIGHTS'] = cleaned_data.groupby(['ORIGIN', 'FL_DATE', 'DEPARTURE_HOUR'])['ORIGIN'].transform('count') # ORIGIN_HOURLY_FLIGHTS - How many flights departed from that origin airport in that hour

cleaned_data['ROUTE_HOURLY_FLIGHTS'] = cleaned_data.groupby(['ROUTE', 'FL_DATE', 'DEPARTURE_HOUR'])['ROUTE'].transform('count') # ROUTE_HOURLY_FLIGHTS - How many flights operated on that route in that hour

cleaned_data['DEST_HOURLY_FLIGHTS'] = cleaned_data.groupby(['DEST', 'FL_DATE', 'DEPARTURE_HOUR'])['DEST'].transform('count') # ROUTE_HOURLY_FLIGHTS - How many flights are at the destination at that hour

In [162]:
cleaned_data['DATE'] = cleaned_data['FL_DATE'].dt.date

In [163]:
from sklearn.preprocessing import LabelEncoder
import joblib

In [164]:
cleaned_data[['OP_UNIQUE_CARRIER', 'ORIGIN_STATE_ABR', 'DEST_STATE_ABR']].head()

,OP_UNIQUE_CARRIER,ORIGIN_STATE_ABR,DEST_STATE_ABR
0,AA,NY,CA
1,AA,CA,NY
2,AA,WI,NC
3,AA,NC,MO
4,AA,MO,NC


In [165]:
le_carrier = LabelEncoder()
le_origin_state = LabelEncoder()
le_dest_state = LabelEncoder()

cleaned_data['OP_UNIQUE_CARRIER'] = le_carrier.fit_transform(cleaned_data['OP_UNIQUE_CARRIER'])
cleaned_data['ORIGIN_STATE_ABR'] = le_origin_state.fit_transform(cleaned_data['ORIGIN_STATE_ABR'])
cleaned_data['DEST_STATE_ABR'] = le_dest_state.fit_transform(cleaned_data['DEST_STATE_ABR'])

In [166]:
joblib.dump(le_carrier, '../encodings/le_carrier.pkl')
joblib.dump(le_origin_state, '../encodings/le_origin_state.pkl')
joblib.dump(le_dest_state, '../encodings/le_dest_state.pkl')

['../encodings/le_dest_state.pkl']

In [167]:
y = cleaned_data["DELAYED"]

In [168]:
columns = cleaned_data.corr(numeric_only=True, method='spearman')['DELAYED'].sort_values(ascending=False)
columns

DELAYED                  1.000000
DEP_DEL15                0.980188
DEP_DELAY_NEW            0.807794
ARR_DEL15                0.745245
DEP_DELAY                0.703921
ARR_DELAY                0.648016
LATE_AIRCRAFT_DELAY      0.636449
CARRIER_DELAY            0.583557
NAS_DELAY                0.346173
WEATHER_DELAY            0.203604
DEPARTURE_HOUR           0.197246
wind_gusts               0.100155
precipitation            0.086713
cloud_cover_low          0.079370
weather_code             0.077780
rain                     0.076795
temperature              0.066343
wind_speed               0.051733
DEPARTURE_MINUTE         0.045239
SECURITY_DELAY           0.044134
snowfall                 0.041412
ORIGIN_HOURLY_FLIGHTS    0.033193
DAY_OF_WEEK              0.028636
MONTH                    0.023176
IS_HOLIDAY_PERIOD        0.019701
DISTANCE                 0.015500
CRS_ELAPSED_TIME         0.014809
DISTANCE_GROUP           0.014103
ORIGIN_STATE_ABR         0.007231
DEST_STATE_ABR

In [169]:
cleaned_data = cleaned_data.drop(columns=['DEP_DELAY_NEW', 'DEP_DELAY', 'LATE_AIRCRAFT_DELAY', 'CARRIER_DELAY', 'NAS_DELAY', 'WEATHER_DELAY', 'SECURITY_DELAY'])

In [170]:
updated_correlation_matrix = cleaned_data.corr(numeric_only=True, method='spearman')
updated_numerical_columns = updated_correlation_matrix['DELAYED'].sort_values(ascending=False)
updated_numerical_columns

DELAYED                  1.000000
DEP_DEL15                0.980188
ARR_DEL15                0.745245
ARR_DELAY                0.648016
DEPARTURE_HOUR           0.197246
wind_gusts               0.100155
precipitation            0.086713
cloud_cover_low          0.079370
weather_code             0.077780
rain                     0.076795
temperature              0.066343
wind_speed               0.051733
DEPARTURE_MINUTE         0.045239
snowfall                 0.041412
ORIGIN_HOURLY_FLIGHTS    0.033193
DAY_OF_WEEK              0.028636
MONTH                    0.023176
IS_HOLIDAY_PERIOD        0.019701
DISTANCE                 0.015500
CRS_ELAPSED_TIME         0.014809
DISTANCE_GROUP           0.014103
ORIGIN_STATE_ABR         0.007231
DEST_STATE_ABR           0.005325
DAY_OF_MONTH             0.001495
OP_CARRIER_FL_NUM        0.000951
ROUTE_HOURLY_FLIGHTS    -0.000489
YEAR                    -0.002175
OP_UNIQUE_CARRIER       -0.008398
DEST_HOURLY_FLIGHTS     -0.036814
CANCELLED     

In [171]:
updated_numerical_columns = updated_numerical_columns.drop(['DELAYED', 'DEPARTURE_MINUTE', 'DISTANCE_GROUP', 'OP_CARRIER_FL_NUM', 'DAY_OF_MONTH', 'YEAR'])
updated_numerical_columns

DEP_DEL15                0.980188
ARR_DEL15                0.745245
ARR_DELAY                0.648016
DEPARTURE_HOUR           0.197246
wind_gusts               0.100155
precipitation            0.086713
cloud_cover_low          0.079370
weather_code             0.077780
rain                     0.076795
temperature              0.066343
wind_speed               0.051733
snowfall                 0.041412
ORIGIN_HOURLY_FLIGHTS    0.033193
DAY_OF_WEEK              0.028636
MONTH                    0.023176
IS_HOLIDAY_PERIOD        0.019701
DISTANCE                 0.015500
CRS_ELAPSED_TIME         0.014809
ORIGIN_STATE_ABR         0.007231
DEST_STATE_ABR           0.005325
ROUTE_HOURLY_FLIGHTS    -0.000489
OP_UNIQUE_CARRIER       -0.008398
DEST_HOURLY_FLIGHTS     -0.036814
CANCELLED                     NaN
DIVERTED                      NaN
Name: DELAYED, dtype: float64

In [172]:
features = updated_numerical_columns.index.tolist()

In [173]:
features.append('ORIGIN')
features.append('DEST')
features.append('ROUTE')

In [174]:
features

['DEP_DEL15',
 'ARR_DEL15',
 'ARR_DELAY',
 'DEPARTURE_HOUR',
 'wind_gusts',
 'precipitation',
 'cloud_cover_low',
 'weather_code',
 'rain',
 'temperature',
 'wind_speed',
 'snowfall',
 'ORIGIN_HOURLY_FLIGHTS',
 'DAY_OF_WEEK',
 'MONTH',
 'IS_HOLIDAY_PERIOD',
 'DISTANCE',
 'CRS_ELAPSED_TIME',
 'ORIGIN_STATE_ABR',
 'DEST_STATE_ABR',
 'ROUTE_HOURLY_FLIGHTS',
 'OP_UNIQUE_CARRIER',
 'DEST_HOURLY_FLIGHTS',
 'CANCELLED',
 'DIVERTED',
 'ORIGIN',
 'DEST',
 'ROUTE']

In [175]:
cols = ['ARR_DEL15', 'ARR_DELAY', 'DEP_DEL15', 'CANCELLED', 'DIVERTED']
for col in cols:
    features.remove(col)

In [176]:
features

['DEPARTURE_HOUR',
 'wind_gusts',
 'precipitation',
 'cloud_cover_low',
 'weather_code',
 'rain',
 'temperature',
 'wind_speed',
 'snowfall',
 'ORIGIN_HOURLY_FLIGHTS',
 'DAY_OF_WEEK',
 'MONTH',
 'IS_HOLIDAY_PERIOD',
 'DISTANCE',
 'CRS_ELAPSED_TIME',
 'ORIGIN_STATE_ABR',
 'DEST_STATE_ABR',
 'ROUTE_HOURLY_FLIGHTS',
 'OP_UNIQUE_CARRIER',
 'DEST_HOURLY_FLIGHTS',
 'ORIGIN',
 'DEST',
 'ROUTE']

In [177]:
X = cleaned_data[features]

In [178]:
from sklearn.model_selection import train_test_split

In [179]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=1234)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=1234)

In [180]:
origin_hourly_avg = X_train.groupby(['ORIGIN', 'DEPARTURE_HOUR'])['ORIGIN_HOURLY_FLIGHTS'].mean().to_dict()
dest_hourly_avg = X_train.groupby(['DEST', 'DEPARTURE_HOUR'])['DEST_HOURLY_FLIGHTS'].mean().to_dict()
route_hourly_avg = X_train.groupby(['ROUTE', 'DEPARTURE_HOUR'])['ROUTE_HOURLY_FLIGHTS'].mean().to_dict()

joblib.dump(origin_hourly_avg, '../encodings/origin_hourly_avg.pkl')
joblib.dump(dest_hourly_avg, '../encodings/dest_hourly_avg.pkl')
joblib.dump(route_hourly_avg, '../encodings/route_hourly_avg.pkl')

['../encodings/route_hourly_avg.pkl']

In [181]:
carrier_delay_map = pd.concat([X_train, y_train], axis=1).groupby('OP_UNIQUE_CARRIER')['DELAYED'].mean()
origin_delay_map = pd.concat([X_train, y_train], axis=1).groupby('ORIGIN')['DELAYED'].mean()
route_delay_map = pd.concat([X_train, y_train], axis=1).groupby('ROUTE')['DELAYED'].mean()
dest_delay_map = pd.concat([X_train, y_train], axis=1).groupby('DEST')['DELAYED'].mean()

# CARRIER_DELAY_RATE - percentage of flights that were delayed for a specific airline. 
X_train['CARRIER_DELAY_RATE'] = X_train['OP_UNIQUE_CARRIER'].map(carrier_delay_map)
X_val['CARRIER_DELAY_RATE'] = X_val['OP_UNIQUE_CARRIER'].map(carrier_delay_map)
X_test['CARRIER_DELAY_RATE'] = X_test['OP_UNIQUE_CARRIER'].map(carrier_delay_map)

# ORIGIN_DELAY_RATE - percentage of flights that were delayed departing from a specific airport. 
X_train['ORIGIN_DELAY_RATE'] = X_train['ORIGIN'].map(origin_delay_map)
X_val['ORIGIN_DELAY_RATE'] = X_val['ORIGIN'].map(origin_delay_map)
X_test['ORIGIN_DELAY_RATE'] = X_test['ORIGIN'].map(origin_delay_map)

# ROUTE_DELAY_RATE - percentage of flights that were delayed on a specific route. 
X_train['ROUTE_DELAY_RATE'] = X_train['ROUTE'].map(route_delay_map)
X_val['ROUTE_DELAY_RATE'] = X_val['ROUTE'].map(route_delay_map)
X_test['ROUTE_DELAY_RATE'] = X_test['ROUTE'].map(route_delay_map)

X_train['DEST_DELAY_RATE'] = X_train['DEST'].map(dest_delay_map)
X_val['DEST_DELAY_RATE'] = X_val['DEST'].map(dest_delay_map)
X_test['DEST_DELAY_RATE'] = X_test['DEST'].map(dest_delay_map)

joblib.dump(carrier_delay_map, '../encodings/carrier_delay_map.pkl')
joblib.dump(origin_delay_map, '../encodings/origin_delay_map.pkl')
joblib.dump(dest_delay_map, '../encodings/dest_delay_map.pkl')
joblib.dump(route_delay_map, '../encodings/route_delay_map.pkl')

['../encodings/route_delay_map.pkl']

In [182]:
route_mean = route_delay_map.mean()
X_val['ROUTE_DELAY_RATE'] = X_val['ROUTE_DELAY_RATE'].fillna(route_mean)
X_test['ROUTE_DELAY_RATE'] = X_test['ROUTE_DELAY_RATE'].fillna(route_mean)

In [183]:
dest_mean = dest_delay_map.mean()
X_val['DEST_DELAY_RATE'] = X_val['DEST_DELAY_RATE'].fillna(dest_mean)
X_test['DEST_DELAY_RATE'] = X_test['DEST_DELAY_RATE'].fillna(dest_mean)

In [184]:
from sklearn.preprocessing import TargetEncoder

In [185]:
te_origin = TargetEncoder()
te_dest = TargetEncoder()
te_route = TargetEncoder()

In [ ]:
# Fit transform the training data
X_train['ORIGIN_ENCODED'] = te_origin.fit_transform(X_train[['ORIGIN']], y_train)
X_train['DEST_ENCODED'] = te_dest.fit_transform(X_train[['DEST']], y_train)
X_train['ROUTE_ENCODED'] = te_route.fit_transform(X_train[['ROUTE']], y_train)

X_val['ORIGIN_ENCODED'] = te_origin.transform(X_val[['ORIGIN']])
X_val['DEST_ENCODED'] = te_dest.transform(X_val[['DEST']])
X_val['ROUTE_ENCODED'] = te_route.transform(X_val[['ROUTE']])

X_test['ORIGIN_ENCODED'] = te_origin.transform(X_test[['ORIGIN']])
X_test['DEST_ENCODED'] = te_dest.transform(X_test[['DEST']])
X_test['ROUTE_ENCODED'] = te_route.transform(X_test[['ROUTE']])

X_train = X_train.drop(columns=['ORIGIN', 'DEST', 'ROUTE'])
X_val = X_val.drop(columns=['ORIGIN', 'DEST', 'ROUTE'])
X_test = X_test.drop(columns=['ORIGIN', 'DEST', 'ROUTE'])

In [187]:
print('OP_UNIQUE_CARRIER' in X_train.columns)

True


In [188]:
# Saving the origin, dest, and route target encoders
joblib.dump(te_origin, '../encodings/origin_te.pkl')
joblib.dump(te_dest, '../encodings/dest_te.pkl')
joblib.dump(te_route, '../encodings/route_te.pkl')

['../encodings/route_te.pkl']

In [189]:
# Saving the training, validation, and testing data
X_train.to_parquet('../data/X_train.parquet', index=False)
X_val.to_parquet('../data/X_val.parquet', index=False)
X_test.to_parquet('../data/X_test.parquet', index=False)
y_train.to_frame().to_parquet('../data/y_train.parquet', index=False)
y_val.to_frame().to_parquet('../data/y_val.parquet', index=False)
y_test.to_frame().to_parquet('../data/y_test.parquet', index=False)

In [190]:
X_train.columns.tolist()

['DEPARTURE_HOUR',
 'wind_gusts',
 'precipitation',
 'cloud_cover_low',
 'weather_code',
 'rain',
 'temperature',
 'wind_speed',
 'snowfall',
 'ORIGIN_HOURLY_FLIGHTS',
 'DAY_OF_WEEK',
 'MONTH',
 'IS_HOLIDAY_PERIOD',
 'DISTANCE',
 'CRS_ELAPSED_TIME',
 'ORIGIN_STATE_ABR',
 'DEST_STATE_ABR',
 'ROUTE_HOURLY_FLIGHTS',
 'OP_UNIQUE_CARRIER',
 'DEST_HOURLY_FLIGHTS',
 'CARRIER_DELAY_RATE',
 'ORIGIN_DELAY_RATE',
 'ROUTE_DELAY_RATE',
 'DEST_DELAY_RATE',
 'ORIGIN_ENCODED',
 'DEST_ENCODED',
 'ROUTE_ENCODED']

In [191]:
X_val.columns.tolist()

['DEPARTURE_HOUR',
 'wind_gusts',
 'precipitation',
 'cloud_cover_low',
 'weather_code',
 'rain',
 'temperature',
 'wind_speed',
 'snowfall',
 'ORIGIN_HOURLY_FLIGHTS',
 'DAY_OF_WEEK',
 'MONTH',
 'IS_HOLIDAY_PERIOD',
 'DISTANCE',
 'CRS_ELAPSED_TIME',
 'ORIGIN_STATE_ABR',
 'DEST_STATE_ABR',
 'ROUTE_HOURLY_FLIGHTS',
 'OP_UNIQUE_CARRIER',
 'DEST_HOURLY_FLIGHTS',
 'CARRIER_DELAY_RATE',
 'ORIGIN_DELAY_RATE',
 'ROUTE_DELAY_RATE',
 'DEST_DELAY_RATE',
 'ORIGIN_ENCODED',
 'DEST_ENCODED',
 'ROUTE_ENCODED']

In [192]:
X_test.columns.tolist()

['DEPARTURE_HOUR',
 'wind_gusts',
 'precipitation',
 'cloud_cover_low',
 'weather_code',
 'rain',
 'temperature',
 'wind_speed',
 'snowfall',
 'ORIGIN_HOURLY_FLIGHTS',
 'DAY_OF_WEEK',
 'MONTH',
 'IS_HOLIDAY_PERIOD',
 'DISTANCE',
 'CRS_ELAPSED_TIME',
 'ORIGIN_STATE_ABR',
 'DEST_STATE_ABR',
 'ROUTE_HOURLY_FLIGHTS',
 'OP_UNIQUE_CARRIER',
 'DEST_HOURLY_FLIGHTS',
 'CARRIER_DELAY_RATE',
 'ORIGIN_DELAY_RATE',
 'ROUTE_DELAY_RATE',
 'DEST_DELAY_RATE',
 'ORIGIN_ENCODED',
 'DEST_ENCODED',
 'ROUTE_ENCODED']

In [193]:
X_train.dtypes

DEPARTURE_HOUR             int64
wind_gusts               float64
precipitation            float64
cloud_cover_low            int64
weather_code               int64
rain                     float64
temperature              float64
wind_speed               float64
snowfall                 float64
ORIGIN_HOURLY_FLIGHTS      int64
DAY_OF_WEEK                int32
MONTH                      int32
IS_HOLIDAY_PERIOD          int64
DISTANCE                 float64
CRS_ELAPSED_TIME         float64
ORIGIN_STATE_ABR           int64
DEST_STATE_ABR             int64
ROUTE_HOURLY_FLIGHTS       int64
OP_UNIQUE_CARRIER          int64
DEST_HOURLY_FLIGHTS        int64
CARRIER_DELAY_RATE       float64
ORIGIN_DELAY_RATE        float64
ROUTE_DELAY_RATE         float64
DEST_DELAY_RATE          float64
ORIGIN_ENCODED           float64
DEST_ENCODED             float64
ROUTE_ENCODED            float64
dtype: object

In [194]:
X_val.dtypes

DEPARTURE_HOUR             int64
wind_gusts               float64
precipitation            float64
cloud_cover_low            int64
weather_code               int64
rain                     float64
temperature              float64
wind_speed               float64
snowfall                 float64
ORIGIN_HOURLY_FLIGHTS      int64
DAY_OF_WEEK                int32
MONTH                      int32
IS_HOLIDAY_PERIOD          int64
DISTANCE                 float64
CRS_ELAPSED_TIME         float64
ORIGIN_STATE_ABR           int64
DEST_STATE_ABR             int64
ROUTE_HOURLY_FLIGHTS       int64
OP_UNIQUE_CARRIER          int64
DEST_HOURLY_FLIGHTS        int64
CARRIER_DELAY_RATE       float64
ORIGIN_DELAY_RATE        float64
ROUTE_DELAY_RATE         float64
DEST_DELAY_RATE          float64
ORIGIN_ENCODED           float64
DEST_ENCODED             float64
ROUTE_ENCODED            float64
dtype: object

In [195]:
X_test.dtypes

DEPARTURE_HOUR             int64
wind_gusts               float64
precipitation            float64
cloud_cover_low            int64
weather_code               int64
rain                     float64
temperature              float64
wind_speed               float64
snowfall                 float64
ORIGIN_HOURLY_FLIGHTS      int64
DAY_OF_WEEK                int32
MONTH                      int32
IS_HOLIDAY_PERIOD          int64
DISTANCE                 float64
CRS_ELAPSED_TIME         float64
ORIGIN_STATE_ABR           int64
DEST_STATE_ABR             int64
ROUTE_HOURLY_FLIGHTS       int64
OP_UNIQUE_CARRIER          int64
DEST_HOURLY_FLIGHTS        int64
CARRIER_DELAY_RATE       float64
ORIGIN_DELAY_RATE        float64
ROUTE_DELAY_RATE         float64
DEST_DELAY_RATE          float64
ORIGIN_ENCODED           float64
DEST_ENCODED             float64
ROUTE_ENCODED            float64
dtype: object

In [215]:
X_train.columns.tolist()

['DEPARTURE_HOUR',
 'wind_gusts',
 'precipitation',
 'cloud_cover_low',
 'weather_code',
 'rain',
 'temperature',
 'wind_speed',
 'snowfall',
 'ORIGIN_HOURLY_FLIGHTS',
 'DAY_OF_WEEK',
 'MONTH',
 'IS_HOLIDAY_PERIOD',
 'DISTANCE',
 'CRS_ELAPSED_TIME',
 'ORIGIN_STATE_ABR',
 'DEST_STATE_ABR',
 'ROUTE_HOURLY_FLIGHTS',
 'OP_UNIQUE_CARRIER',
 'DEST_HOURLY_FLIGHTS',
 'CARRIER_DELAY_RATE',
 'ORIGIN_DELAY_RATE',
 'ROUTE_DELAY_RATE',
 'DEST_DELAY_RATE',
 'ORIGIN_ENCODED',
 'DEST_ENCODED',
 'ROUTE_ENCODED']

In [216]:
X_train.head()

,DEPARTURE_HOUR,wind_gusts,precipitation,cloud_cover_low,weather_code,rain,temperature,wind_speed,snowfall,ORIGIN_HOURLY_FLIGHTS,...,ROUTE_HOURLY_FLIGHTS,OP_UNIQUE_CARRIER,DEST_HOURLY_FLIGHTS,CARRIER_DELAY_RATE,ORIGIN_DELAY_RATE,ROUTE_DELAY_RATE,DEST_DELAY_RATE,ORIGIN_ENCODED,DEST_ENCODED,ROUTE_ENCODED
8284982,19,29.2,0.0,0,3,0.0,33.5,9.9,0.0,83,...,1,12,11,0.236848,0.234546,0.279227,0.217983,0.234982,0.216969,0.269276
2333376,17,31.3,0.0,16,2,0.0,29.6,11.3,0.0,23,...,1,0,7,0.238592,0.245921,0.277228,0.214917,0.246289,0.213174,0.267393
4113436,12,31.7,0.0,0,0,0.0,-11.5,13.7,0.0,41,...,3,13,26,0.175199,0.247707,0.266308,0.234389,0.247753,0.233971,0.264486
5527512,9,24.1,0.0,0,3,0.0,10.7,11.6,0.0,4,...,1,7,57,0.175034,0.184154,0.195209,0.199883,0.184315,0.199796,0.195764
7966712,21,18.7,0.0,0,0,0.0,34.0,5.2,0.0,11,...,1,12,21,0.236848,0.171468,0.152717,0.179131,0.173807,0.179481,0.152639
